# 04 — Data quality gate (DQX)

Runs [Databricks DQX](https://databrickslabs.github.io/dqx/) over every `silver_<entity>` table. Valid rows flow on to `silver_<entity>`; rows failing an error-criticality check are split into a parallel `{UC_SCHEMA}_quarantine.<entity>` table so bad data never reaches gold.

The baseline check is key-not-null: every `PRIMARY_KEYS` column must be present. This also catches upstream schema drift, since a renamed or removed source field makes its silver column go null and the row gets quarantined instead of passing silently. Extend it with business rules (date ranges, enum membership, referential checks against your dims); see the DQX docs for the full check catalog.

In [ ]:
# Job parameter bridge: when run as a Databricks Job task, the bundle's base_parameters arrive as
# notebook widgets. Mirror them into os.environ before the setup cell reads them, so the same
# os.getenv(...) logic works as a job, in a workspace, or locally via .env. dbutils is absent under
# local Databricks Connect, so the except clause handles that case.
import os
for _k in ("UC_CATALOG", "UC_SCHEMA"):
    try:
        _v = dbutils.widgets.get(_k)            # noqa: F821 — dbutils is injected in Databricks
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass

In [ ]:
# Dual-mode setup: runs locally via Databricks Connect and inside a Databricks Git Folder.
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
try:
    spark  # already present inside a Databricks workspace notebook
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "main")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "synergy_baseball_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
B, S, G = f"{UC_CATALOG}.{BRONZE_SCHEMA}", f"{UC_CATALOG}.{SILVER_SCHEMA}", f"{UC_CATALOG}.{GOLD_SCHEMA}"
print("target:", B, "|", S, "|", G)

In [ ]:
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F
from synergy_schemas import ENTITIES, PRIMARY_KEYS

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}_quarantine")
QUAR = f"{UC_CATALOG}.{UC_SCHEMA}_quarantine"
engine = DQEngine(WorkspaceClient())

def baseline_checks(entity):
    # Every natural-key column must be non-null (error criticality).
    return [{"name": "keys_not_null", "criticality": "error",
             "check": {"function": "is_not_null", "for_each_column": list(PRIMARY_KEYS[entity])}}]

for e in ENTITIES:
    entity = e["name"]
    tbl = f"{S}.silver_{entity}"
    # An entity may have no silver table on a given run: it landed 0 rows (e.g. scoped to a roster with
    # no data in the window) or its source pull errored (transient API 5xx), so its bronze/silver table
    # is absent or empty. Skip those rather than fail the whole DQ gate.
    if entity not in PRIMARY_KEYS:
        print(f"  silver_{entity:24} — no PRIMARY_KEYS defined; skipping")
        continue
    if not spark.catalog.tableExists(tbl):
        print(f"  silver_{entity:24} — no silver table (nothing landed upstream); skipping")
        continue
    checks = baseline_checks(entity)
    status = DQEngine.validate_checks(checks)
    assert not status.has_errors, f"{entity}: invalid checks: {status}"
    valid, quar = engine.apply_checks_by_metadata_and_split(spark.table(tbl), checks)
    # Quarantine only error-criticality failures; warn-only rows stay in silver.
    quar = quar.filter(F.col("_errors").isNotNull()) if quar is not None else None
    n_quar = 0
    if quar is not None:
        quar.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{QUAR}.{entity}")
        n_quar = spark.table(f"{QUAR}.{entity}").count()
    # Clean rows flow back to silver, minus DQX's bookkeeping columns.
    valid.drop("_errors", "_warnings").write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl)
    print(f"  silver_{entity:24} clean={spark.table(tbl).count():>6}  quarantined={n_quar}")
print("\nGold (notebook 05) reads the clean silver tables; the _quarantine schema is the DQ audit trail.")